## Script for histograms creation

In [1]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib
import yaml


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression, save_resolutions_to_yaml
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT(90)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

In [8]:
base_path = "/workspace/root_notebooks/results"

################ SETTINGS ##################################
analysis_dir = "kl_pm_control_sample"
analysis_hash = "7b6c187f3e4b0390d51437281fd4292b"
img_dir = "img"
root_file_name = f"tag_ks_pm_chann_div_1D_truth_variables.root"
eff_graph_file_name = "efficiency_purity_graphs.root"

mode_chann = "channels"
single_channel_draw = "pi+pi-pi+pi-"

if mode_chann == "channels":
  flags = channName.values()
else:
  flags = ["has000", "no000", "Data", "MC sum"]
  
mode = "zoom"
nocut_hash = "nocuts"
log_flag = False
fit_double_gauss = True
normality_max_abs_skewness = 0.3
normality_max_abs_excess_kurtosis = 0.5
draw_core_tail = True

hist_config_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_1D.yaml"
with open(hist_config_file_path, "r") as f:
    hist_config = yaml.safe_load(f)
############################################################

hist_file_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{root_file_name}"
eff_graph_file_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{eff_graph_file_name}"

#################### ANALYSIS REGISTRY #####################
import pandas as pd
analysis_registry = f"{base_path}/{analysis_dir}/{analysis_hash}/analysis_registry.csv"
df = pd.read_csv(analysis_registry)

hash_available = df["hash"]

## Making histograms

In [6]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

# Set up variables 1D
variables_1D_truth = set(hist_config.get("truth_only_vars_1d", []))
variables_1D = [hist for hist in hist_config.get("histograms_1d", []).keys()
                if hist not in variables_1D_truth]

#################### OBJECT TO FIND ################################
histograms = {}

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        continue
    
    histograms[hash] = {}

    for var in variables_1D_truth:
      dir_obj = hist_file.Get(f"{hash}/{var}")
      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for flag in flags:    
        hist_name = f"{var}_{flag}_{mode}"
        if hist_name not in [key.GetName() for key in dir_obj.GetListOfKeys()]:
            continue
    
        histograms[hash][var][flag] = dir_obj.Get(hist_name).Clone()

hist_file.Close()

In [9]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{analysis_hash}/{img_dir}/resolutions"
os.makedirs(img_dir_path, exist_ok=True)
ROOT.gStyle.SetHistTopMargin(0)

################## DRAW USING THSTACK ###########################
canvases = []
legends = []
stacks = []
funcs = []
par_dict = {}

for hash in histograms:
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for i, var in enumerate(histograms[hash]):
        fit_func = None
        fit_result = None
        fit_model = None
        func1 = None
        func2 = None
        path_var = f"{path_hash}/{var}"
        os.makedirs(path_var, exist_ok=True)
        
        legend = ROOT.TLegend(0.5, 0.8, 0.84875, 0.9)
        legend.SetTextFont(62)
        legend.SetTextSize(0.025)
        
        # Dodajemy hash do nazwy, aby uniknąć konfliktów w pamięci ROOTa przy wielu folderach
        stack = ROOT.THStack(f"{var}_stack_{hash}", "")
        
        for flag in histograms[hash][var]:

            if single_channel_draw and flag != single_channel_draw:
                continue
            
            hist = histograms[hash][var][flag]
            if hist:

              if log_flag:
                hist.SetMinimum(0.1)
              
              if single_channel_draw:
                hist.SetMaximum(hist.GetBinContent(hist.GetMaximumBin()) * 1.2)

              if fit_double_gauss:
                skewness = hist.GetSkewness()
                excess_kurtosis = hist.GetKurtosis()
                use_single_gauss = (
                    abs(skewness) < normality_max_abs_skewness
                    and abs(excess_kurtosis) < normality_max_abs_excess_kurtosis
                )

                if use_single_gauss:
                  fit_model = "single_gaussian"
                  fit_func = ROOT.TF1(f"{var}_fit_{hash}_{flag}", "gaus", hist.GetXaxis().GetXmin(), hist.GetXaxis().GetXmax())
                  fit_func.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
                  fit_func.SetParNames("A", "mean", "sigma")
                  fit_func.SetParLimits(0, 0, hist.GetMaximum() * 100)
                  fit_func.SetParLimits(1, hist.GetMean() - hist.GetRMS(), hist.GetMean() + hist.GetRMS())
                  fit_func.SetParLimits(2, 0, hist.GetRMS() * 2)
                  fit_label = "Gaussian Fit"
                else:
                  fit_model = "double_gaussian"
                  fit_func = ROOT.TF1(f"{var}_fit_{hash}_{flag}", ROOT.double_gaussian_cpp, hist.GetXaxis().GetXmin(), hist.GetXaxis().GetXmax(), 5)
                  fit_func.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS() * 0.5, hist.GetRMS() * 2.0, 0.7)
                  fit_func.SetParNames("A1", "mean", "sigma1", "sigma2", "f")
                  fit_func.SetParLimits(0, 0, hist.GetMaximum() * 100)
                  fit_func.SetParLimits(1, hist.GetMean() - hist.GetRMS(), hist.GetMean() + hist.GetRMS())
                  fit_func.SetParLimits(2, 0, hist.GetRMS() * 2)
                  fit_func.SetParLimits(3, hist.GetRMS(), hist.GetRMS() * 4)
                  fit_func.SetParLimits(4, 0, 1)
                  fit_label = "Double Gaussian Fit"

                fit_func.SetLineColor(ROOT.kBlack)
                fit_func.SetLineStyle(ROOT.kDashed)
                
                fit_result = hist.Fit(fit_func, "ISR")

                if int(fit_result) == 0 and fit_model == "double_gaussian":
                  func1 = ROOT.TF1("core_gaussian", ROOT.single_gauss_comp1, histograms[hash][var][flag].GetXaxis().GetXmin(), histograms[hash][var][flag].GetXaxis().GetXmax(), 4)
                  func1.SetParameters(fit_result.Parameter(0), fit_result.Parameter(1), fit_result.Parameter(2), fit_result.Parameter(4))

                  func1.SetLineColor(ROOT.kViolet)
                  func1.SetLineStyle(ROOT.kDashed)
                  
                  func2 = ROOT.TF1("tail_gaussian", ROOT.single_gauss_comp2, histograms[hash][var][flag].GetXaxis().GetXmin(), histograms[hash][var][flag].GetXaxis().GetXmax(), 4)
                  func2.SetParameters(fit_result.Parameter(0), fit_result.Parameter(1), fit_result.Parameter(3), fit_result.Parameter(4))

                  func2.SetLineColor(ROOT.kBlue)
                  func2.SetLineStyle(ROOT.kDashed)
                  
                  funcs.append(func1)
                  funcs.append(func2)
                  funcs.append(fit_func)


              hist.SetLineColor(channColor[flag])
                
              if flag != "Data":
                # Opcję rysowania przekazujemy bezpośrednio w metodzie Add dla THStack
                stack.Add(hist, "hist")

                legend.AddEntry(hist, channTitle[flag], "l")

                if fit_double_gauss and fit_func is not None:
                  legend.AddEntry(fit_func, fit_label, "l")
                  if draw_core_tail and func1 and func2:
                    legend.AddEntry(func1, "Core Gaussian", "l")
                    legend.AddEntry(func2, "Tail Gaussian", "l")

              else:
                hist.SetMarkerStyle(20)
                hist.SetMarkerColor(channColor[flag])
                stack.Add(hist, "pe1")
                legend.AddEntry(hist, channTitle[flag], "pe1")
                            
        canvas = ROOT.TCanvas(f"{var}_canvas_{hash}", f"{var} Canvas", 600, 600)
        
        stack.Draw("NOSTACK")
        if fit_double_gauss and fit_func is not None:
            fit_func.Draw("same")
            if draw_core_tail and func1 and func2:
                func1.Draw("same")
                func2.Draw("same")

            pave = ROOT.TPaveText(0.15, 0.55, 0.43, 0.9, "NDC")

            # Konfiguracja wyglądu pudełka (opcjonalnie)
            pave.SetFillColor(0)       # Białe tło (0 = przezroczyste/białe w zależności od stylu)
            pave.SetBorderSize(1)      # Cienka ramka (0 = brak ramki)
            pave.SetTextFont(42)       # Zwykła czcionka (lub 62 dla bolda)
            pave.SetTextSize(0.023)
            pave.SetTextAlign(12)      # Wyrównanie tekstu: 1=lewa, 2=środek w pionie

            # 2. Dodajemy kolejne linijki tekstu
            pave.AddText(fit_label)

            if fit_result is not None and int(fit_result) == 0:
                mean, mean_err = fit_result.Parameter(1), fit_result.ParError(1)
                chi2_ndf = fit_result.Chi2() / fit_result.Ndf() if fit_result.Ndf() != 0 else float('nan')
                real_var = var.removesuffix("_resolution")

                if fit_model == "single_gaussian":
                    sigma, sigma_err = fit_result.Parameter(2), fit_result.ParError(2)
                    par_dict[real_var] = {
                        "mean": [mean, mean_err],
                        "sigma_core": [sigma, sigma_err],
                        "sigma_tail": [sigma, sigma_err],
                        "amplitude_ratio": [1.0, 0.0],
                        "fit_model": fit_model,
                        "fit_status": int(fit_result),
                        "chi2_ndf": chi2_ndf,
                    }
                    pave.AddText(f'A = {fit_result.Parameter(0):.2g} #pm {fit_result.ParError(0):.2g}')
                    pave.AddText(f'#mu = ({mean:.2g} #pm {mean_err:.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                    pave.AddText(f'#sigma = ({sigma:.2g} #pm {sigma_err:.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                else:
                    sigma_core, sigma_core_err = fit_result.Parameter(2), fit_result.ParError(2)
                    sigma_tail, sigma_tail_err = fit_result.Parameter(3), fit_result.ParError(3)
                    amplitude_ratio, amplitude_ratio_err = fit_result.Parameter(4), fit_result.ParError(4)
                    par_dict[real_var] = {
                        "mean": [mean, mean_err],
                        "sigma_core": [sigma_core, sigma_core_err],
                        "sigma_tail": [sigma_tail, sigma_tail_err],
                        "amplitude_ratio": [amplitude_ratio, amplitude_ratio_err],
                        "fit_model": fit_model,
                        "fit_status": int(fit_result),
                        "chi2_ndf": chi2_ndf,
                    }
                    pave.AddText(f'A_{{core}} = {fit_result.Parameter(0):.2g} #pm {fit_result.ParError(0):.2g}')
                    pave.AddText(f'#mu = ({mean:.2g} #pm {mean_err:.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                    pave.AddText(f'#sigma_{{core}} = ({sigma_core:.2g} #pm {sigma_core_err:.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                    pave.AddText(f'#sigma_{{tail}} = ({sigma_tail:.2g} #pm {sigma_tail_err:.2g}) {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}')
                    pave.AddText(f'#frac{{A_{{core}}}}{{A_{{tail}}}} = {amplitude_ratio:.2g} #pm {amplitude_ratio_err:.2g}')
            else:
                pave.AddText("Fit failed")

            # 3. Rysujemy całe pudełko na płótnie jednym poleceniem!
            pave.Draw()
        
        if stack.GetHistogram():
            resolution = stack.GetHistogram().GetXaxis().GetBinWidth(1)

            title_x = f'{hist_config.get("histograms_1d", {}).get(var, {}).get("title", "")} [{hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}]'
            title_y = f'Counts / {resolution:.2f} {hist_config.get("histograms_1d", {}).get(var, {}).get("unit", "")}'
            
            stack.GetHistogram().GetXaxis().SetTitle(title_x)
            stack.GetHistogram().GetYaxis().SetTitle(title_y)

        ####### Set log y ########
        if log_flag:
          stack.SetMinimum(0.1)
          stack.SetMaximum(stack.GetMaximum() * 50)
        else:
          stack.SetMinimum(0)
                
        canvas.SetLogy(log_flag)
        ##########################

        legend.Draw()

        canvas.Modified()
        canvas.Update()
        canvas.Draw()
        canvas.SaveAs(f"{path_var}/{var}_{mode}_logy_{log_flag}.pdf")
        canvas.Close()

    res_config_dir = os.path.join(os.path.dirname(hist_config_file_path), hash)
    os.makedirs(res_config_dir, exist_ok=True)
    save_resolutions_to_yaml(par_dict, f"{res_config_dir}/resolutions_config.yaml")

        

Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/kl_pm_control_sample/config/nocuts/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/kl_pm_control_sample/config/21f528cb/resolutions_config.yaml
Pomyślnie zrzucono wyniki rozdzielczości do pliku: /workspace/root_notebooks/results/kl_pm_control_sample/config/eff9a201/resolutions_config.yaml
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      15012.1
NDf                       =          295
Edm                       =  8.34755e-07
NCalls                    =          169
A1                        =      30511.9   +/-   58.7075      	 (limited)
mean                      =     -1.61589   +/-   0.0059796    	 (limited)
sigma1                    =      2.02102   +/-   0.0070404    	 (limited)
sigma2                    =      8.21501   +/-   0.0556187    	 (limited)
f                         =     0

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/resolutions/nocuts/Delta_t_fit_resolution/Delta_t_fit_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/resolutions/nocuts/R_KS_resolution/R_KS_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/resolutions/nocuts/Delta_t_MC/Delta_t_MC_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/resolutions/nocuts/R_KL_resolution/R_KL_resolution_zoom_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/kl_pm_control_sample/7b6c187f3e4b0390d51437281fd4292b/img/resolutio

## Calculating events, efficiencies and purities

In [1]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)

#################### OBJECT TO FIND ################################
event_counts = {}
cuts_list = {}
efficiencies = {}
purities = {}

#################### EXTRACT EVENT COUNTS #######################
for hash in hash_available:
    obj = hist_file.Get(f"{hash}/ChannelIntegrals")
    obj_cut_list = hist_file.Get(f"{hash}/CutList")

    if obj:
      chann_integrals_str = obj.GetTitle()
      event_counts[hash] = json.loads(chann_integrals_str)

    if obj_cut_list:
      raw_title = obj_cut_list.GetTitle()
      cleaned_str = raw_title.strip("[] ").replace('"', '')
      cuts_list[hash] = [c.strip() for c in cleaned_str.split(',') if c.strip()]

hist_file.Close()
#################################################################

#################### CALCULATE EFFICIENCIES AND PURITIES #######################
for hash in hash_available:
  if hash not in event_counts:
     continue

  efficiencies[hash] = {}
  purities[hash] = {}

  mc_sum = int(event_counts[hash].get("MC sum", 0))

  for flag in flags:
      total_events = int(event_counts.get(nocut_hash, None).get(flag, 0))
      passed_events = int(event_counts[hash].get(flag, 0))
      efficiencies[hash][flag] = passed_events / total_events if total_events > 0 else 0
      purities[hash][flag] = passed_events / mc_sum if mc_sum > 0 else 0
###############################################################################

################## PRINT RESULTS ##################
for hash in hash_available:
    if hash not in efficiencies:
        continue
    print(f"Results for cut list: {cuts_list.get(hash, [])}")
    for flag in flags:
        event_count = int(event_counts.get(hash, {}).get(flag, 0))
        mc_sum = int(event_counts.get(hash, {}).get("MC sum", 0))
        eff = efficiencies[hash].get(flag, 0)
        pur = purities[hash].get(flag, 0)
        print(f"  Flag: {flag}, Event Count: {event_count / 10**6:.2f} x 10^6, Efficiency: {eff * 100:.2f}%, Purity: {pur * 100:.2f}%, (MC - Data) / Data: {100*(mc_sum - event_count) / event_count if event_count > 0 else 0:.2f}%")
    print()




NameError: name 'ROOT' is not defined